# MiniLAr + MLP

In this notebook, we design and train a Multi-Layer Perceptron (MLP) for a small LArTPC particle-image classification task. The dataset is **MiniLAr**: MNIST-shaped 28x28 grayscale images built from tiny LArTPC particle projections. The labels are particle species rather than handwritten digits.

MiniLAr is intentionally MNIST-like: each item is an image tensor plus one integer label. This lets us reuse the standard PyTorch image-classification workflow while keeping the example closer to DUNE and LArTPC reconstruction.

## Goals
1. Get familiar with the MiniLAr dataset
2. Build a pipeline to stream data for SGD
3. Design an MLP and train it on MiniLAr

Let's start with usual import!

In [ ]:
# Import drawing tools, set style
import matplotlib.pyplot as plt
import matplotlib as mpl
%matplotlib inline

mpl.rcParams['figure.figsize'] = [8, 6]
mpl.rcParams['font.size'] = 16
mpl.rcParams['axes.grid'] = True

# Import numpy and torch, set default device based on GPU availability
import numpy as np
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# Set the seed to always get the same result from RNG calls in this notebook
SEED = 12345
np.random.seed(SEED)    # Setting the seed for reproducibility
torch.manual_seed(SEED) # This is how you do for torch!

## 1. MiniLAr Dataset

MNIST is widely used in introductory machine learning courses because it gives a simple image-classification task with a very convenient `Dataset` interface. MiniLAr follows the same idea, but the images are small LArTPC particle projections and the labels are particle classes.

The dataset wrapper behaves like `torchvision.datasets.MNIST`: it knows how to find or fetch the raw file, split it into train/test samples, and return one `(image, label)` tuple at a time.

In [ ]:
from dataset.lartpc_particles import LArTPCParticles

# Specify where you want the dataset to land
LOCAL_DATA_DIR = 'dataset/minilar-dataset'

# Fill this in once the dataset file is hosted.
# You can also use a local path, e.g. DATA_URL='/path/to/lartpc_particles.pt'
DATA_URL = 'https://s3df.slac.stanford.edu/data/neutrino/dune-tech-analysis/minilar/raw/lartpc_particles.pt'

# Create the dataset. If DATA_URL is set, download/process into LOCAL_DATA_DIR.
dataset = LArTPCParticles(
    LOCAL_DATA_DIR,
    train=True,
    download=DATA_URL is not None,
    url=DATA_URL,
)

print('Using data cache:', LOCAL_DATA_DIR)

MiniLAr is a PyTorch `Dataset`. All torch `Dataset` instances have two useful and common methods: the length method and data element access via index.

In [ ]:
# Length of the MiniLAr dataset
print( len(dataset)  )

# Type of the first element in the MiniLAr dataset
print( type(dataset[0]) )

How each data element is presented depends on the particular `Dataset` implementation. In MiniLAr, as in MNIST, it is a tuple of length 2: **data** and **label**.

In [ ]:
# Access a specific entry in the MiniLAr dataset
ENTRY=0
data, label = dataset[ENTRY]

# Show the type of data and the type of label associated with it
print('Type of data  :', type(data),  'shape', data.shape)
print('Type of label :', type(label), 'value', label)

# Use matplotlib to draw the data as an image
data = data.view(data.shape[1:])
plt.imshow(data,cmap='gray')
plt.show()

### Prompt

Look at the image and the integer label. What visual information do you think a human would use to tell particle classes apart: length, brightness, scattering, showeriness, or something else?

In [ ]:
# How do we find out what the classes mean?!
# ...

<a href="dataloader"></a>
## 2. Streaming MiniLAr data using `DataLoader`

PyTorch provides a generalized tool to interface an iterable data instance called `DataLoader`. This is useful for streaming data in a typical ML workflow.

1. iterable batching of data
2. custom data sampling, random provided by default
3. custom data pre-processing
4. parallel data reading

Let's play a little bit.

In [ ]:
from torch.utils.data import DataLoader

# Define basic dataset of 10 integers from 0 to 9
# It fits the definition of a torch dataset because it has
# a __len__ and a __getitem__ method
data = np.arange(10)

# Initialize a basic sequential dataloader, check how the values are fetched
print('\nVanilla data loader')
loader = DataLoader(data)
for index, batch_data in enumerate(loader):
    print(index, batch_data)

# Batch the data by groups of 3 (batch_size=3), check how the values are fetched
print('\n... batch_size set to 3')
loader = DataLoader(data, batch_size=3)
for index, batch_data in enumerate(loader):
    print(index,batch_data)

# Now drop the last batch if the length is < 3, check how the values are fetched
print('\n... + drop_last set to True')
loader = DataLoader(data, batch_size=3, drop_last=True)
for index, batch_data in enumerate(loader):
    print(index,batch_data)

# Finally, shuffle the input randomly because batching it
print('\n... + shuffle set to True')
loader = DataLoader(data, batch_size=3, drop_last=True, shuffle=True)
for index, batch_data in enumerate(loader):
    print(index,batch_data)

We can wrap the data loading loop by another loop so that we can continue iterating our dataset indefinitely for optimizing our model over **1 epoch** (full dataset loop). But if you want a reproducibility of the data subsets created by `DataLoader`, you can set the random seed explicitly as well.

In [ ]:
# Define data loader with batch_size 3, dropping the last batch if batch_size < 3, with random shuffle
loader = DataLoader(data, batch_size=3, drop_last=True, shuffle=True)

# Loop over the full dataset twice, shuffling each time!
for i in range(2):
    print('Loop', i)
    for index, batch_data in enumerate(loader):
        print(index, batch_data)

# Now manually set the seed each time to ensure the batches are loaded the same way at each epoch
print('\nRepeating by setting the seed')
for i in range(2):
    print('Loop',i)
    torch.manual_seed(1)
    for index, batch_data in enumerate(loader):
        print(index,batch_data)

### 2.1 Creating your own dataset

For your research, often you will want to create our own dataset with handy utilities. You can define your own dataset and use with the `DataLoader` as long as it's either iterable (i.e. implements `__iter__` built-in method) or supports `len` function and an index access, i.e. `__len__` and `__getitem__` support.

Let's create a trivial example:

In [ ]:
import time

class toy_dataset:
    """Toy dataset which range from 0 to 99."""

    def __init__(self):
        """Initialize a dataset of 100 integers ranging from 0 to 99."""
        self._data = tuple(range(100))

    def __len__(self):
        """Returns length of the dataset.

        Returns
        -------
        int
            Length of the dataset
        """
        return len(self._data)

    def __getitem__(self, index):
        """Sleeps for a bit and than returns the requested element

        Parameters
        ----------
        index : int
            Index of the element in the dataset

        Returns
        -------
        int
            Value of the data at the requested index
        """
        time.sleep(0.1)
        return self._data[index]

# Initialize the dataset
data = toy_dataset()

# Build a load based on it with batch_size 10
loader = DataLoader(data, batch_size=10, shuffle=True)

# Loop over your toy dataset, print the data!
for index, batch_data in enumerate(loader):
    print(index, batch_data)

... which is very slow because our dataset takes 0.1 second per data element access, or 1 second per batch :( While we put `time.sleep(0.1)` intentionally, this (=slow-down for per-sample data yield) can happen in reality. Perhaps you are forced to read each data sample from a file-system (i.e. "disk") each time because your data is too big to be stored in memory, or maybe you have to pre-process your data with complicated procedures. 

There is a simple way to speed this up this by setting another `DataLoader` constructor argument: `num_workers`.

In [ ]:
loader = DataLoader(data, batch_size=10, shuffle=True, num_workers=5)

# for index, batch_data in enumerate(loader):
#     print(index, batch_data)
    

By setting `num_workers=5`, we told the `DataLoader` to use 5 separate _workers_ (processes) each responsible for fetching data in parallel. This is one of the simplest way to parallelize your data streaming.  In this tutorial, we don't cover a multi-GPU nor multi-node-multi-GPU data distribution. But tools are provided for those as well (e.g. Pytorch provides `DistributeDataParallel`). 

### 2.2 Creating `DataLoader` with the MiniLAr dataset

Now let's create a MiniLAr data loader with `batch_size=64`, `shuffle=True`, `num_workers=4`:

In [ ]:
loader = torch.utils.data.DataLoader(dataset,
                                     batch_size=64,
                                     shuffle=True,
                                     num_workers=5,
                                     pin_memory=False)

The above introduces only 1 additional argument: `pin_memory=True`. This can speed up data transfer to GPU, at the cost of some resource overhead. Read [here](https://devblogs.nvidia.com/how-optimize-data-transfers-cuda-cc/) for more details. If you are not sure about the details, set to `True` when using GPU.

So let's play with it! First of all, it has the concept of "length".

In [ ]:
print('length of DataLoader:',len(loader))
print('By the way, batch size * length =', loader.batch_size * len(loader))

The length of a `DataLoader` is the number of batches. The batch size is the number of images per batch, so `batch_size * len(loader)` is approximately the number of samples seen in one epoch.

In [ ]:
# Create an iterator which can cycle multiple times through the same dataloader
from itertools import cycle
iter = cycle(loader)

# Loop over 10 entries in the loader (10 batches)
for i in range(10):
    # Fetch a batch
    batch = next(iter)

    # Print the iteration number
    print('Iteration',i)

    # Print the particle labels for that batch
    print(batch[1])

... and this is how `data` looks like:

In [ ]:
print('Shape of an image batch data', batch[0].shape)

... which is naturally a batch of 64 grayscale images of size 28x28.

### Prompt

Before we train anything: what information is lost when we flatten a 28x28 image into a vector of 784 numbers? Why might that matter for particle images?

## 3. MiniLAr classification using MLP

Let's try classifying LArTPC particle images using an MLP.

### 3.1 Model definition
We follow a similar approach to the previous notebook: a linear layer maps the input image to hidden activations, a non-linear activation bends the model, and a final linear layer maps to particle-class logits.

### Prompt

A batch has shape `[N, 1, 28, 28]`. Which dimension counts images, which counts channels, and which dimensions are spatial? What will happen when the MLP flattens the image?

In [ ]:
NUM_CLASSES = 5

class MLP(torch.nn.Module):
    """Basic multi-layer perceptron with two layers and a
    LeakyReLU function in between.
    """
    
    def __init__(self, num_filters=16):
        """Initialize the linear layers and the non-linear activation function.

        Parameters
        ----------
        num_filters : int, default 16
            Number of hidden neurons in between the two linear layers
        """
        # Initialize the underlying torch.nn.Module parent class
        super().__init__()
        
        # MLP w/ one hidden layer
        self._classifier = torch.nn.Sequential(
            torch.nn.Linear(28*28, num_filters), # Maps all pixels onto a hidden layer
            torch.nn.LeakyReLU(), # Use LeakyReLU as an activation
            torch.nn.Linear(num_filters, NUM_CLASSES) # Map the hidden activations onto 5 possible outputs
        )

    def forward(self, x):
        """Pass one batch of data through the model.

        Parameters
        ----------
        x : torch.Tensor
            (N, 1, 28, 28) N MiniLAr images of size 28*28

        Returns
        -------
        torch.Tensor
            (N) Logit output of the last linear layer (one number per image)
        """
        # Flatten the image as a 1D array (to be fed to the MLP)
        x_1d = x.view(-1, np.prod(x.size()[1:]))

        # Pass through the MLP
        return self._classifier(x_1d)

### Prompt

The final layer has one output score per particle class. Before applying softmax, these scores are called logits. Why might we prefer to train on logits directly instead of manually converting them to probabilities first?


### 3.2 Train loop function
Next, let's define a train and test loop function. This is pretty much copied from the last notebook.

In [ ]:
# Python time tracking package
import time

# Python CSV writing package
import csv

# Progress bar package, makes waiting easier!
# If you're curious, tqdm comes from the arabic word for progress, taqadum
from tqdm import tqdm

def run_train(model, loader,  
              num_iterations=100, log_dir='logs_mlp', log_prefix=None,
              optimizer='SGD', lr=0.001, device=None):
    """Train loop function.
    
    Parameters
    ----------
    model : torch.nn.Module
        Machine learning model
    loader : torch.DataLoader
        Batch loading tool
    num_iterations : int, default 100
        Number of iterations to run the optimization over
    log_dir : str, default 'logs_mlp'
        Path to the log directory
    log_prefix : str, optional
        Prefix to prepend onto log names
    optimizer : str, default 'SGD'
        Name of the optimizer as defined here: https://docs.pytorch.org/docs/stable/optim.html
    lr : float, default 0.001
        Learning rate
    """
    # Create the log directory if it does not exist
    !mkdir -p {log_dir}

    # Define log name
    prefix = '' if log_prefix is None else f'{log_prefix}_'
    train_log_name = f'{log_dir}/{prefix}train.csv'

    # Initialize the loss function (cross-entropy loss)
    criterion = torch.nn.CrossEntropyLoss()

    # Initialize the optimizer
    optimizer_fn = getattr(torch.optim,optimizer)
    optimizer = optimizer_fn(model.parameters(), lr=lr)

    # Initialize a progress bar
    pbar = tqdm(total=num_iterations, position=0, leave=True)

    # Loop over the entire dataset until the requested number of iterations is reached
    iteration = 0
    with open(train_log_name, 'w') as csvfile:
        # Initialize CSV writer
        writer = csv.writer(csvfile)
        writer.writerow(['iter', 'epoch', 'loss'])
        while iteration < num_iterations:
            # Loop over data in the loader (one epoch)
            for data, label in loader:
                # If a device is specify, move data/label to it
                if device is not None:
                    data, label = data.to(device), label.to(device)

                optimizer.zero_grad()
    
                loss = criterion(model(data), label)
                
                loss.backward()
                optimizer.step()
                
                # Append the log
                writer.writerow([iteration, iteration/len(loader), loss.item()])
    
                # Increment iteration count, update the progress bar
                iteration += 1
                pbar.update(1)

                # Break if the necessary number of iterations is reached
                if iteration >= num_iterations:
                    break

### Train!

Let's train for 4000 steps using Adam optimizer. The number of hidden neurons is default = 16.

### Prompt

What do you expect an MLP to learn from these images? Is it using local image structure, or only the flattened pixel values?

In [ ]:
# Use the device selected at the top of the notebook.
print('Training on device:', device)

# Initialize an MLP with 16 hidden neurons (place on the correct device!)
model_a = MLP(16).to(device=device)

# Run the train loop
run_train(model_a, loader, 10000, log_dir='logs_mlp', optimizer='Adam', device=device)

If your machine is slow, skip over this training process and simply load the model parameters from an earlier training process!

The way this file was produced was by simply calling
```python
torch.save(model_a.state_dict(), 'weights/mlp_model_a.ckpt')
```

In [ ]:
# torch.save(model_a.state_dict(), 'weights/mlp_model_a.ckpt')

In [ ]:
# state_dict = torch.load('weights/mlp_model_a.ckpt', weights_only=True)
# model_a.load_state_dict(state_dict)

Now we look at the train log, we can use the one trained earlier to save time...

In [ ]:
import pandas as pd

df = pd.read_csv('logs_mlp/train.csv')
# df = pd.read_csv('logs_mlp/pretrain.csv')
df

In [ ]:
df.rolling(10).mean().plot('epoch', 'loss', grid=True)

### 3.3 Exercise A

Repeat the same training for `model_b` = `MLP` with 32 filters. 

In [ ]:
# Your code here!

### 3.4 Exercise B

1. Repeat the exercise A and measure how long it takes in wall-time.
2. Repeat again, but this time with GPU enabled, and measure how long it takes.
3. How  much gain do we observe on a simple MLP like this?

In [ ]:
# Your code here!

In [ ]:
# Your code here!

### Running on test dataset
Both models seem to be trained OK. Let's benchmark their performance using the test dataset. The dataset wrapper uses `train=False` to load the held-out split, analogous to the usual MNIST train/test split.

In [ ]:
# Use prepared data handler
# train=False will load the test set
test_dataset = LArTPCParticles(
    LOCAL_DATA_DIR,
    train=False,
    download=DATA_URL is not None,
    url=DATA_URL,
)

test_loader = torch.utils.data.DataLoader(test_dataset,
                                          batch_size=64,
                                          shuffle=False,
                                          num_workers=4,
                                          pin_memory=False)

You can see there are fewer samples in the test split than in the training split.

In [ ]:
len(test_loader), len(test_loader)*test_loader.batch_size

### Inference loop function

Let's now write a function to run the inference. This would be similar to the training loop. A key difference is to use a scope `with torch.set_grad_enabled(False)` which disables gradient calculation and caching of intermediate data for it. This results in less memory usage, so you should do this when you run your model for inference and not training.

In [ ]:
def run_test(model, loader, device=None):
    """Runs the trained model on test data.

    Parameters
    ----------
    model : torch.nn.Module
        Model definition
    loader : torch.DataLoader
        Method to load the test data
    device : str, optional
        Device on which to put the data/model if one uses a GPU

    Returns
    -------
    torch.Tensor
        (N) List of labels for each image in the test set
    torch.Tensor
        (N, 5) Softmax scores predicted for each of the 5 particle classes
    """
    # Initialize the output lists
    label_v, softmax_v = [],[]

    # Initialize the softmax function, s_i = exp(-v_i)/(sum_i exp(-v_i))
    softmax   = torch.nn.Softmax(dim=1)

    # Initialize a progress bar
    pbar = tqdm(total=len(loader), position=0, leave=True)

    # Loop over test data, append labels and softmax predictions
    with torch.set_grad_enabled(False):
        for data, label in loader:
            if device is not None:
                data, label = data.to(device), label.to(device)
            label_v.append  ( label.detach().reshape(-1)   )
            softmax_v.append( softmax(model(data)).detach())
            pbar.update(1)

    # Return
    return torch.concat(label_v).cpu().numpy(), torch.concat(softmax_v).cpu().numpy()

In [ ]:
# Let's run the inference!
model_a.to(device=device)
labels, scores = run_test(model_a, test_loader, device=device)

In [ ]:
# The output was cached in case you are running on a slow machine

In [ ]:
# torch.save({'labels': labels, 'scores': scores}, 'inference/mlp_output.pkl')

Once again, if your machine is slow, I have done this for you upstream, simply load the information!

In [ ]:
# saved_data = torch.load('inference/mlp_output.pkl', weights_only=False)
# labels, scores = saved_data['labels'], saved_data['scores']

Now let's look at performance. First we can see how often we get things right. Predictions show up as 5 numbers per image, each representing a **score** associated with one possible particle class.

In [ ]:
scores

We convert scores to specific predictions by picking the most likely particle class for each image:

In [ ]:
preds = np.argmax(scores, axis=1)
preds

Now we can compare to the labels...

In [ ]:
np.sum(labels == preds)/len(labels)

The accuracy is a useful first summary, but it does not tell us which particles are confused with which.

Let me do you one better and create a confusion matrix:

In [ ]:
# Define a function which estimates the confusion matrix of the model.
def make_confusion_matrix(labels, preds):
    """Make a row-normalized confusion matrix."""
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=float)
    for label, pred in zip(labels, preds):
        cm[int(label), int(pred)] += 1

    row_sums = cm.sum(axis=1, keepdims=True)
    return np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums > 0)


def plot_confusion_matrix(cm, class_names):
    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(cm, vmin=0, vmax=1, cmap='viridis')
    fig.colorbar(image, ax=ax, label='fraction of true class')
    ax.grid(False)

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_yticklabels(class_names)
    ax.set_xlabel('Prediction')
    ax.set_ylabel('True label')

    for row in range(cm.shape[0]):
        for col in range(cm.shape[1]):
            value = cm[row, col]
            color = 'black' if value > 0.5 else 'white'
            ax.text(col, row, f'{value:.2f}', ha='center', va='center', color=color)

    fig.tight_layout()
    return fig, ax


cm = make_confusion_matrix(labels, preds)
plot_confusion_matrix(cm, dataset.class_names);

### 3.5 Exercise C
1. Run the inference for `model_a` and `model_b`.
2. Using the results, compute the accuracy over the whole test dataset for both models.
3. Which particle classes are easiest or hardest to classify? Use the confusion matrix to justify your answer.

In [ ]:
# Your code here!

In [ ]:
# Your code here!

### 3.6 Exercise D

1. Count the number of parameters in our `model_a`
2. How about `model_b`?


In [ ]:
# Your code here!

In [ ]:
# Your code here!